In [ ]:

%pip install -q -U "crewai[google-genai,tools]" ddgs


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/

In [ ]:

import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is missing. Add it in Colab Secrets "
        "and enable notebook access."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
print("Gemini API key configured successfully.")


Gemini API key configured successfully.


In [ ]:

from crewai import LLM

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.2,
    max_tokens=4096
)

print("Gemini LLM configured.")


Gemini LLM configured.


In [ ]:

from crewai.tools import tool
from ddgs import DDGS

@tool("DuckDuckGo Search")
def duckduckgo_search(query: str) -> str:
    """Search the web and return relevant results with titles,
    snippets, and source URLs for a given query."""

    try:
        results = DDGS().text(query, max_results=5)

        if not results:
            return "No search results found for this query."

        output = []

        for index, result in enumerate(results, start=1):
            output.append(
                f"Result {index}\n"
                f"Title: {result.get('title', 'N/A')}\n"
                f"Snippet: {result.get('body', 'N/A')}\n"
                f"URL: {result.get('href', 'N/A')}"
            )

        return "\n\n".join(output)

    except Exception as e:
        return f"Search failed: {type(e).__name__}: {e}"

print("DuckDuckGo search tool created.")


DuckDuckGo search tool created.


In [ ]:

from crewai import Agent
from crewai_tools import FileWriterTool

file_writer = FileWriterTool()

# Agent 1: Researcher
researcher = Agent(
    role="Research Specialist",
    goal=(
        "Research the given topic using web search and collect "
        "relevant facts, useful insights, and source URLs."
    ),
    backstory=(
        "You are a careful research specialist. You distinguish "
        "evidence from assumptions and never invent sources."
    ),
    tools=[duckduckgo_search],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

# Agent 2: Writer
writer = Agent(
    role="Professional Blog Writer",
    goal=(
        "Write an engaging blog post of approximately 600 words "
        "using the research provided."
    ),
    backstory=(
        "You are a professional content writer who turns research "
        "into clear, engaging, well-structured articles."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3
)

# Agent 3: Editor
editor = Agent(
    role="Blog Editor",
    goal=(
        "Edit the draft for readability, tone, structure, and "
        "factual consistency, and save the final article."
    ),
    backstory=(
        "You are a meticulous editor. Preserve supported facts, "
        "remove unsupported claims, improve clarity, and use "
        "the file-writing tool to save the final Markdown article."
    ),
    tools=[file_writer],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3
)

print("All three agents created.")


All three agents created.


In [ ]:

from crewai import Task

research_task = Task(
    description=(
        "Research the topic: {topic}\n\n"
        "Use DuckDuckGo search to investigate the topic. "
        "Collect the key facts, recent developments where relevant, "
        "useful examples, and at least three relevant source URLs "
        "when available. Return organized research notes and clearly "
        "distinguish verified information from uncertainty."
    ),
    expected_output=(
        "Organized research notes containing key findings, useful "
        "examples, and source titles and URLs. Do not invent citations."
    ),
    agent=researcher
)

writing_task = Task(
    description=(
        "Write a blog post of approximately 600 words about {topic}, "
        "using the research supplied by the preceding task. Include "
        "an engaging title, introduction, logical section headings, "
        "examples where useful, and a conclusion. Preserve source URLs "
        "for factual claims and avoid unsupported statistics."
    ),
    expected_output=(
        "A complete blog post draft of approximately 600 words in "
        "Markdown, with a title, headings, conclusion, and source links."
    ),
    agent=writer
)

editing_task = Task(
    description=(
        "Edit the draft produced by the preceding task about {topic}. "
        "Improve grammar, readability, tone, organization, and factual "
        "consistency. Preserve useful source links. Do not invent facts "
        "or claim that a statement is verified without supporting evidence. "
        "Save the final edited article to blog_post.md using FileWriterTool. "
        "The file must contain the article itself, not a discussion of the task."
    ),
    expected_output=(
        "The final edited Markdown blog post, with a clear title, "
        "well-organized sections, conclusion, and relevant source links. "
        "The file blog_post.md must be written successfully."
    ),
    agent=editor,
    output_file="blog_post.md"
)

print("All three tasks created.")


All three tasks created.


In [ ]:

from crewai import Crew, Process

blog_crew = Crew(
    agents=[
        researcher,
        writer,
        editor
    ],
    tasks=[
        research_task,
        writing_task,
        editing_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Blog Post Crew assembled.")


Blog Post Crew assembled.


In [ ]:

TOPIC = input("Enter the topic for your blog post: ").strip()

if not TOPIC:
    print("Please enter a valid topic.")
else:
    result = await blog_crew.kickoff_async(
        inputs={"topic": TOPIC}
    )

    print("\nBlog post generated successfully!")
    print("Topic:", TOPIC)

Enter the topic for your blog post: indian cricket performance 2026


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a0ff0665-d406-44e8-86a7-a57d7e697e39                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: indian cricket performance 2026                                                      │
│                                                                                                                 │
│  Use DuckDuckGo search to investigate the topic. Collect the key facts, recent developments where relevant,     │
│  useful examples, and at least three relevant source URLs when available. Return organized research notes and   │
│  clearly distinguish verified information from uncertainty.                                                     │
│  ID: a4f0a7bb-6b39-4326-ba7d-c3a987b351cb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research the topic: indian cricket performance 2026                                                      │
│                                                                                                                 │
│  Use DuckDuckGo search to investigate the topic. Collect the key facts, recent developments where relevant,     │
│  useful examples, and at least three relevant source URLs when available. Return organized research notes and   │
│  clearly distinguish verified information from uncertainty.                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: duck_duck_go_search                                                                                      │
│  Args: {'query': 'indian cricket performance 2026'}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Blog Editor                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Inside the 2026 Season: Squad Depth, Schedule, and the Evolution of Indian Cricket                           │
│                                                                                                                 │
│  As the international cricket calendar unfolds, the performance and trajectory of the Indian cricket team       │
│  continue to capture the attention of fans worldwide. Navigating a rigorous multi-format schedule requires      │
│  more than just star power—it demands exceptional squad depth, strategic foresight, and a seamless pipeline     │
│  from youth setups to the senior international stage.                                                           │
│                                                                                                                 │
│  While live scores and win-loss records shift dynamically with every match played, the architectural blueprint  │
│  of Indian cricket remains clear. Systematic planning by the Board of Control for Cricket in India (BCCI)       │
│  ensures the team is consistently tested across home and away fixtures, keeping the squad battle-ready for all  │
│  conditions.                                                                                                    │
│                                                                                                                 │
│  Let’s examine what defines the Indian cricket landscape in 2026, highlighting upcoming marquee tours and the   │
│  underlying factors driving team stability.                                                                     │
│                                                                                                                 │
│  ## The 2026 Home Calendar: Key Series to Watch                                                                 │
│                                                                                                                 │
│  The backbone of any successful international team lies in its home fixtures, and the 2026 calendar offers a    │
│  compelling lineup designed to test India's tactical adaptability across formats.                               │
│                                                                                                                 │
│  ### 1. The Afghanistan Tour of India (June 2026)                                                               │
│  Marking a significant mid-year highlight, the Afghanistan cricket team is scheduled to tour India in June      │
│  2026. As officially outlined in the BCCI's home calendar scheduling, this fixture is set to feature a          │
│  comprehensive contest comprising **one Test match and three One Day International (ODI) matches** (Source:     │
│  [Afghan cricket team in India in 2026](https://en.wikipedia.org/wiki/Afghan_cricket_team_in_India_in_2026)).   │
│  This series provides a valuable platform to evaluate classical red-ball endurance alongside dynamic            │
│  white-ball strategies.                                                                                         │
│                                                                                                                 │
│  ### 2. The West Indies Tour of India (September – October 2026)                                                │
│  As the autumn season kicks off, the action continues w

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Edit the draft produced by the preceding task about indian cricket performance 2026. Improve grammar,    │
│  readability, tone, organization, and factual consistency. Preserve useful source links. Do not invent facts    │
│  or claim that a statement is verified without supporting evidence. Save the final edited article to            │
│  blog_post.md using FileWriterTool. The file must contain the article itself, not a discussion of the task.     │
│  Agent: Blog Editor                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Blog post generated successfully!
Topic: indian cricket performance 2026


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a0ff0665-d406-44e8-86a7-a57d7e697e39                                                                       │
│  Final Output: # Inside the 2026 Season: Squad Depth, Schedule, and the Evolution of Indian Cricket             │
│                                                                                                                 │
│  As the international cricket calendar unfolds, the performance and trajectory of the Indian cricket team       │
│  continue to capture the attention of fans worldwide. Navigating a rigorous multi-format schedule requires      │
│  more than just star power—it demands exceptional squad depth, strategic foresight, and a seamless pipeline     │
│  from youth setups to the senior international stage.                                                           │
│                                                                                                                 │
│  While live scores and win-loss records shift dynamically with every match played, the architectural blueprint  │
│  of Indian cricket remains clear. Systematic planning by the Board of Control for Cricket in India (BCCI)       │
│  ensures the team is consistently tested across home and away fixtures, keeping the squad battle-ready for all  │
│  conditions.                                                                                                    │
│                                                                                                                 │
│  Let’s examine what defines the Indian cricket landscape in 2026, highlighting upcoming marquee tours and the   │
│  underlying factors driving team stability.                                                                     │
│                                                                                                                 │
│  ## The 2026 Home Calendar: Key Series to Watch                                                                 │
│                                                                                                                 │
│  The backbone of any successful international team lies in its home fixtures, and the 2026 calendar offers a    │
│  compelling lineup designed to test India's tactical adaptability across formats.                               │
│                                                                                                                 │
│  ### 1. The Afghanistan Tour of India (June 2026)                                                               │
│  Marking a significant mid-year highlight, the Afghanistan cricket team is scheduled to tour India in June      │
│  2026. As officially outlined in the BCCI's home calendar scheduling, this fixture is set to feature a          │
│  comprehensive contest comprising **one Test match and three One Day International (ODI) matches** (Source:     │
│  [Afghan cricket team in India in 2026](https://en.wikipedia.org/wiki/Afghan_cricket_team_in_India_in_2026)).   │
│  This series provides a valuable platform to evaluate classical red-ball endurance alongside dynamic            │
│  white-ball strategies.                                                                                         │
│                                                                                                                 │
│  ### 2. The West Indies Tour of India (September – October 2026)                                                │
│  As the autumn season kicks off, the action continues 

In [ ]:

from pathlib import Path
from IPython.display import display, Markdown

output_path = Path("blog_post.md")

if output_path.exists() and output_path.stat().st_size > 0:
    final_blog = output_path.read_text(encoding="utf-8")

    print(f"File created: {output_path.resolve()}")
    print(f"File size: {output_path.stat().st_size} bytes")
    print(f"Word count: {len(final_blog.split())}")

    display(Markdown(final_blog))
else:
    print("The output file was not created or is empty.")
    print("Inspect the crew execution logs for errors.")


File created: /content/blog_post.md
File size: 4794 bytes
Word count: 571


# Cultivating the Future: How Artificial Intelligence is Transforming Modern Farming

Agriculture is currently undergoing its most significant evolution since the invention of the tractor. As the global population grows and environmental challenges mount, traditional farming methods are receiving a high-tech upgrade. Artificial Intelligence (AI) has emerged as a primary catalyst for modern precision farming, transforming a historically resource-intensive industry into a data-driven, highly optimized system. 

By integrating machine learning, computer vision, Internet of Things (IoT) sensors, and autonomous robotics, today's farmers are achieving unprecedented levels of efficiency. According to recent research on [artificial intelligence in modern agriculture](https://www.sciencedirect.com/science/article/pii/S2589721726000917), these technologies are fundamentally reshaping how food is grown, managed, and harvested.

## Smarter Soil and Crop Monitoring

At the heart of AI-driven agriculture is the ability to make sense of massive amounts of data. Machine learning algorithms analyze historical data, local weather patterns, and high-resolution satellite or drone imagery. This synthesis of information provides farmers with deep insights into soil health, moisture levels, and nutrient deficiencies long before problems become visible to the naked eye. 

Furthermore, recent advancements in [artificial intelligence in agriculture](https://www.sciencedirect.com/science/article/pii/S2666154325001334) highlight how real-time edge computing allows AI models to process IoT sensor data locally on the farm. This means farmers can receive instant feedback without needing constant, high-speed cloud connectivity out in the field.

## Precision Weed and Pest Control

Gone are the days of blanketing entire fields in chemical treatments. Thanks to computer vision, AI-enabled cameras and drones can now spot weeds, crop diseases, and pests in real time. 

Instead of applying herbicides or pesticides uniformly, smart sprayers target only the affected plants. This precision spraying drastically reduces chemical usage, lowers operating costs, and minimizes environmental runoff, promoting a much healthier farm ecosystem.

## Autonomous Machinery and Labor Solutions

Labor shortages have long challenged the agricultural sector, but autonomous machinery and robotics are stepping in to fill the gap. AI-guided tractors, weeders, and harvesting robots can operate around the clock with incredible accuracy. 

Whether they are delicately picking ripe fruits or precisely planting seeds, these robotic systems handle labor-intensive tasks reliably. This not only mitigates the impact of labor shortages but also ensures that crops are harvested at their optimal time.

## Predictive Analytics for Risk Management

Farming has always involved navigating weather and market volatility, but predictive analytics are helping farmers manage these uncertainties. Producers use predictive models to forecast market demand, determine optimal harvest times, and anticipate weather-related risks like sudden frosts or droughts. By anticipating these challenges, growers can better protect their yields and secure stronger operational outcomes.

## Ecosystem Sustainability and the Road Ahead

Ultimately, the promise of AI in farming lies in balance. Recent studies emphasize that modern agricultural technology must balance high productivity with environmental protection. By addressing water scarcity and combating soil nutrient depletion through targeted resource regulation, AI is helping pave the way for sustainable food production. 

As explored in depth by [Springer's research on AI technologies in agriculture](https://link.springer.com/chapter/10.1007/978-981-95-1268-3_8), the fusion of machine learning and traditional farming is a present-day reality driving the industry forward. By embracing these smart systems, farmers are equipped to feed a growing population while helping to preserve natural resources for generations to come.

---

## Sources and References

1. **ScienceDirect** – *Artificial intelligence in modern agriculture: recent advances...*  
   URL: [https://www.sciencedirect.com/science/article/pii/S2589721726000917](https://www.sciencedirect.com/science/article/pii/S2589721726000917)
2. **ScienceDirect** – *Artificial intelligence in agriculture: Advancing crop productivity and...*  
   URL: [https://www.sciencedirect.com/science/article/pii/S2666154325001334](https://www.sciencedirect.com/science/article/pii/S2666154325001334)
3. **Springer** – *AI Technologies in Agriculture: Transforming Farming with...*  
   URL: [https://link.springer.com/chapter/10.1007/978-981-95-1268-3_8](https://link.springer.com/chapter/10.1007/978-981-95-1268-3_8)

In [ ]:
from google.colab import files
files.download("blog_post.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>